# 第 22 课：NumPy IO

NumPy 可以读写专用二进制格式 `.npy`、多个数组组成的 `.npz`，以及逗号分隔等文本文件。二进制格式能保留 dtype 和 shape，文本格式便于与其他软件交换。

In [1]:
import tempfile
from pathlib import Path
import numpy as np

work = Path(tempfile.mkdtemp())
print('临时目录：', work)

临时目录： C:\Users\17526\AppData\Local\Temp\tmpegox2_ma


## 1. `save` 和 `load` 保存单个数组

`np.save` 写入 `.npy` 文件，`np.load` 读取它。文件名没有扩展名时，NumPy 通常会自动添加 `.npy`。

In [2]:
a = np.arange(6).reshape(2, 3)
npy_file = work / 'array.npy'
np.save(npy_file, a)
loaded = np.load(npy_file)
print('读取结果：\n', loaded)
print('dtype：', loaded.dtype, 'shape：', loaded.shape)
print('是否相等：', np.array_equal(a, loaded))

读取结果：
 [[0 1 2]
 [3 4 5]]
dtype： int64 shape： (2, 3)
是否相等： True


## 2. `savez` 保存多个数组

`np.savez` 把多个数组放入 `.npz` 压缩包式容器。使用关键字参数可以给数组命名，读取后通过字典式下标访问。需要节省空间时可使用 `np.savez_compressed`。

In [ ]:
x = np.arange(5)
y = np.sin(x)
npz_file = work / 'arrays.npz'
np.savez(npz_file, numbers=x, sine=y)#键值对，键的名称
archive = np.load(npz_file)
print('文件中的名称：', archive.files)
print('numbers：', archive['numbers'])
print('sine：', archive['sine'])
archive.close()

文件中的名称： ['numbers', 'sine']
numbers： [0 1 2 3 4]
sine： [ 0.          0.84147098  0.90929743  0.14112001 -0.7568025 ]


## 3. `savetxt` 和 `loadtxt` 文本读写

文本文件可以用 `delimiter` 指定分隔符，用 `fmt` 控制输出格式。读取时要使用相同的分隔符，并可用 `dtype` 指定类型。

In [ ]:
table = np.array([[1.2, 3.4], [5.6, 7.8]])
txt_file = work / 'table.csv'
np.savetxt(txt_file, table, fmt='%.1f', delimiter=',', header='x,y', comments='')#header表示第一行的表头，comments表示表头前面的东西，不写默认为#
table_loaded = np.loadtxt(txt_file, delimiter=',', skiprows=1)#skiprows表示跳过第一行
print('文本读取结果：\n', table_loaded)#打印时不会把逗号打印出来

文本读取结果：
 [[1.2 3.4]
 [5.6 7.8]]


## 4. `genfromtxt` 处理缺失值

`loadtxt` 适合规则完整的数据；文本中有缺失值时，`genfromtxt` 可以指定 `missing_values` 和 `filling_values`。实际项目中还要注意编码、表头和列类型。

In [5]:
missing_file = work / 'missing.csv'
missing_file.write_text('1,2\n3,\n5,6\n', encoding='utf-8')
filled = np.genfromtxt(missing_file, delimiter=',', filling_values=-1)
print('缺失值填充后：\n', filled)

缺失值填充后：
 [[ 1.  2.]
 [ 3. -1.]
 [ 5.  6.]]


## 本节练习

创建一个 3×3 数组：1. 保存为 `.npy` 并读取验证；2. 将数组和它的平方保存到 `.npz`；3. 保存为逗号分隔文本，再用 `loadtxt` 读回。

In [11]:
ar = np.array(range(9)).reshape(3,3)
np.save(work / 'exersize',ar)
np.savez(work / 'exersize.npz',num=ar,s=ar**2)
np.savetxt(work / 'exersize.csv',ar,fmt='%d',delimiter=',')
print(np.load(work / 'exersize.npy' ))
print(np.load(work / 'exersize.npz' ))
np.load(work / 'exersize.npz' ).close()
print(np.loadtxt(work / 'exersize.csv',delimiter=','))


[[0 1 2]
 [3 4 5]
 [6 7 8]]
NpzFile 'C:\\Users\\17526\\AppData\\Local\\Temp\\tmpegox2_ma\\exersize.npz' with keys: num, s
[[0. 1. 2.]
 [3. 4. 5.]
 [6. 7. 8.]]


In [6]:
data = np.arange(1, 10).reshape(3, 3)
np.save(work / 'exercise.npy', data)
np.savez(work / 'exercise.npz', data=data, square=data ** 2)
np.savetxt(work / 'exercise.csv', data, fmt='%d', delimiter=',')
print(np.load(work / 'exercise.npy'))
with np.load(work / 'exercise.npz') as z:
    print('npz keys：', z.files)
print(np.loadtxt(work / 'exercise.csv', delimiter=','))

[[1 2 3]
 [4 5 6]
 [7 8 9]]
npz keys： ['data', 'square']
[[1. 2. 3.]
 [4. 5. 6.]
 [7. 8. 9.]]


### 小结

`.npy` 适合单个数组，`.npz` 适合多个数组，`savetxt/loadtxt` 适合可读文本。读取不可信的对象数组时不要随意启用 `allow_pickle=True`。

学这些主要是为了让你算出来的数据能够下次接着用，或者读取别人给你的数据。数组只放在程序变量里，程序结束后就没了；保存到文件后才能留住结果。

举个实际场景：你有一份每天的气温数据，要计算平均值。

temperatures = np.loadtxt("temperatures.csv", delimiter=",")

average = np.mean(temperatures)

如果计算结果还要留着：

np.save("temperatures.npy", temperatures)

下次直接读取：

temperatures = np.load("temperatures.npy")